## 1. Imports

In [1]:
import os
import numpy as np
from PIL import Image, ImageFile
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

## 2. Image Preprocessing

In [2]:
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    "data/train",
    target_size=(96, 96),
    batch_size=32,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    "data/test",
    target_size=(96, 96),
    batch_size=32,
    class_mode='binary'
)

Found 19995 images belonging to 2 classes.
Found 5000 images belonging to 2 classes.


## 3. Load and Build Model Architecture

In [3]:
base_model = MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])


## 4. Compile and Train Model

In [4]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

model.fit(
    train_generator,
    epochs=3,
    validation_data=test_generator
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_96 (Functional)     │ (None, 3, 3, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/3
514/625 ━━━━━━━━━━━━━━━━━━━━ 30s 279ms/step - accuracy: 0.9296 - loss: 0.1614

C:\Users\Super\miniconda\envs\myenv\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


625/625 ━━━━━━━━━━━━━━━━━━━━ 227s 349ms/step - accuracy: 0.9437 - loss: 0.1356 - val_accuracy: 0.9466 - val_loss: 0.1350
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 181s 290ms/step - accuracy: 0.9630 - loss: 0.0925 - val_accuracy: 0.9496 - val_loss: 0.1231
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 204s 326ms/step - accuracy: 0.9732 - loss: 0.0698 - val_accuracy: 0.9492 - val_loss: 0.1333


## 5. Evaluation

In [5]:
loss, accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {accuracy:.2f}")

model.save("cats_dogs_model.h5")

print("Training complete.")

157/157 ━━━━━━━━━━━━━━━━━━━━ 30s 190ms/step - accuracy: 0.9492 - loss: 0.1333


Test Accuracy: 0.95
Training complete.


## 6. Testing with random images

In [6]:
model = load_model("cats_dogs_model.h5")

for i in range(1, 9):
    img_path = f"test_data/test_image_{i}.jpg"
    print(f"\nTesting: {img_path}")

    img = image.load_img(img_path, target_size=(96, 96))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0][0]

    if prediction > 0.5:
        label = "Dog"
        confidence = prediction
    else:
        label = "Cat"
        confidence = 1 - prediction

    print(f"Prediction: {label}")
    print(f"Confidence: {confidence * 100:.2f}%")


Testing: test_data/test_image_1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Prediction: Dog
Confidence: 99.96%

Testing: test_data/test_image_2.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
Prediction: Dog
Confidence: 99.99%

Testing: test_data/test_image_3.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
Prediction: Cat
Confidence: 100.00%

Testing: test_data/test_image_4.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Prediction: Cat
Confidence: 100.00%

Testing: test_data/test_image_5.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
Prediction: Cat
Confidence: 98.99%

Testing: test_data/test_image_6.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Prediction: Cat
Confidence: 98.58%

Testing: test_data/test_image_7.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Prediction: Dog
Confidence: 100.00%

Testing: test_data/test_image_8.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Prediction: Dog
Confidence: 100.00%
